# 03. 담당 예측기, YOLO v1 손실, 11-point AP

## 학습 목표

정답과 IoU가 가장 큰 예측기를 책임자로 선택하고, YOLO v1 다항 손실을 구성 요소별로 계산합니다. 마지막에는 점수 임계값 하나로 판단하지 않고 precision-recall 곡선과 VOC 2007 방식 11-point AP를 계산합니다.

## 실행 방법

`python -m pip install jupyter numpy` 후 위에서 아래로 실행하세요. 이 노트북은 교육용 NumPy 구현이며 자동미분 학습 코드가 아닙니다. 실무 학습에는 PyTorch/TensorFlow 연산으로 같은 계약을 옮기고 공식 구현과 수치 대조가 필요합니다.

In [ ]:
import numpy as np  # 벡터화한 IoU, 손실, 누적 평가 계산을 위해 NumPy를 가져옵니다.

LAMBDA_COORD = 5.0  # 논문이 좌표 손실을 강화하기 위해 사용한 λ_coord 값입니다.
LAMBDA_NOOBJ = 0.5  # 객체가 없는 예측의 신뢰도 손실을 줄이는 λ_noobj 값입니다.
EPSILON = 1e-12  # 0으로 나누기와 수치적 불안정을 피하기 위한 매우 작은 양수입니다.

## 1. IoU가 가장 큰 박스 예측기를 책임자로 정하기

YOLO v1은 객체가 있는 셀의 B개 예측 중 정답과 현재 IoU가 가장 큰 하나만 좌표 예측을 책임지게 합니다. 이 특수화가 없으면 여러 예측기가 같은 목표를 똑같이 따라가려 할 수 있습니다.

In [ ]:
def center_to_corners(boxes):  # [cx,cy,w,h] 배열을 [x1,y1,x2,y2] 배열로 변환합니다.
    boxes = np.asarray(boxes, dtype=np.float64)  # 입력을 안정적인 실수 배열로 통일합니다.
    if boxes.shape[-1] != 4:  # 마지막 축에 네 좌표가 있는지 확인합니다.
        raise ValueError("박스 마지막 차원은 4여야 합니다.")  # 잘못된 텐서 형태를 즉시 알립니다.
    if np.any(boxes[..., 2:] <= 0.0):  # 폭이나 높이가 0 이하인 박스가 있는지 검사합니다.
        raise ValueError("폭과 높이는 양수여야 합니다.")  # 이후 제곱근과 IoU 계산이 의미를 갖도록 강제합니다.

    half_size = boxes[..., 2:] / 2.0  # 중심에서 모서리까지의 반 폭과 반 높이를 계산합니다.
    return np.concatenate([boxes[..., :2] - half_size, boxes[..., :2] + half_size], axis=-1)  # 왼쪽 위와 오른쪽 아래를 이어 반환합니다.


def pairwise_iou_with_target(predicted_boxes, target_box):  # 여러 예측과 정답 하나 사이의 IoU를 계산합니다.
    predicted_corners = center_to_corners(predicted_boxes)  # 예측들을 교집합 계산에 편한 모서리 좌표로 바꿉니다.
    target_corners = center_to_corners(target_box)  # 정답도 같은 좌표 형식으로 바꿉니다.
    top_left = np.maximum(predicted_corners[:, :2], target_corners[:2])  # 브로드캐스팅으로 예측별 교집합 시작점을 구합니다.
    bottom_right = np.minimum(predicted_corners[:, 2:], target_corners[2:])  # 예측별 교집합 끝점을 구합니다.
    intersection_size = np.maximum(bottom_right - top_left, 0.0)  # 겹치지 않는 축의 음수 길이를 0으로 제한합니다.
    intersection = intersection_size[:, 0] * intersection_size[:, 1]  # 각 예측의 교집합 넓이를 계산합니다.
    predicted_area = predicted_boxes[:, 2] * predicted_boxes[:, 3]  # 정규화 좌표에서 각 예측 넓이는 w×h입니다.
    target_area = target_box[2] * target_box[3]  # 정답 넓이도 같은 방식으로 계산합니다.
    union = predicted_area + target_area - intersection  # 합집합 넓이는 두 넓이 합에서 교집합을 뺍니다.
    return intersection / np.maximum(union, EPSILON)  # 안전한 나눗셈으로 예측별 IoU를 반환합니다.


def select_responsible_predictor(predicted_boxes, target_box):  # 정답과 가장 잘 겹치는 예측기의 인덱스와 IoU를 고릅니다.
    ious = pairwise_iou_with_target(predicted_boxes, target_box)  # B개 예측의 IoU를 한 번에 계산합니다.
    responsible_index = int(np.argmax(ious))  # argmax로 최고 IoU 위치를 구하고 Python 정수로 변환합니다.
    return responsible_index, ious  # 선택 결과와 진단용 전체 IoU를 함께 반환합니다.


target_box = np.array([0.52, 0.48, 0.30, 0.24], dtype=np.float64)  # 한 셀이 담당하는 정답 박스를 정의합니다.
predicted_boxes = np.array(  # 같은 셀의 두 박스 예측을 정의합니다.
    [[0.50, 0.50, 0.28, 0.25], [0.68, 0.44, 0.22, 0.20]],  # 첫 예측이 정답과 더 잘 겹치도록 구성합니다.
    dtype=np.float64,  # 모든 좌표를 실수로 저장합니다.
)
responsible_index, predictor_ious = select_responsible_predictor(predicted_boxes, target_box)  # 책임 예측기를 선택합니다.
print("예측기별 IoU:", predictor_ious)  # 두 예측의 품질을 나란히 출력합니다.
print("책임 예측기:", responsible_index)  # 0부터 시작하는 선택 인덱스를 출력합니다.

assert responsible_index == 0  # 이 예제에서는 첫 번째 예측기가 책임자가 되어야 합니다.

## 2. 다섯 손실 항을 분리해 계산하기

아래 함수는 객체가 하나 있는 셀을 대상으로 합니다.

- `xy`: 책임 예측기의 중심 좌표 오차
- `wh`: 큰 박스 오차가 과도하게 지배하지 않도록 폭·높이의 제곱근 오차
- `object_confidence`: 책임 예측기의 신뢰도를 IoU 목표에 맞추는 오차
- `no_object_confidence`: 책임자가 아닌 예측기의 신뢰도를 0으로 낮추는 오차
- `classification`: 객체가 있는 셀의 조건부 클래스 확률 오차

원 논문 수식은 전체 `S×S` 셀에 대한 합입니다. 여기서는 한 셀의 동작을 투명하게 보기 위해 범위를 줄였습니다.

In [ ]:
def yolo_v1_object_cell_loss(predicted_boxes, predicted_confidences, predicted_classes, target_box, target_class):  # 객체 셀 하나의 손실을 항목별로 계산합니다.
    predicted_boxes = np.asarray(predicted_boxes, dtype=np.float64)  # 박스 예측을 (B,4) 실수 배열로 통일합니다.
    predicted_confidences = np.asarray(predicted_confidences, dtype=np.float64)  # B개 신뢰도를 실수 배열로 통일합니다.
    predicted_classes = np.asarray(predicted_classes, dtype=np.float64)  # C개 조건부 클래스 값을 실수 배열로 통일합니다.
    target_box = np.asarray(target_box, dtype=np.float64)  # 정답 박스를 실수 배열로 통일합니다.
    target_class = np.asarray(target_class, dtype=np.float64)  # 원-핫 정답 클래스를 실수 배열로 통일합니다.

    if predicted_boxes.ndim != 2 or predicted_boxes.shape[1] != 4:  # 박스가 정확히 (B,4)인지 검사합니다.
        raise ValueError("predicted_boxes의 모양은 (B, 4)여야 합니다.")  # 축 해석 오류를 막습니다.
    if predicted_confidences.shape != (predicted_boxes.shape[0],):  # 신뢰도 수가 박스 수 B와 같은지 검사합니다.
        raise ValueError("박스마다 신뢰도 하나가 필요합니다.")  # 박스-신뢰도 대응이 깨진 입력을 거부합니다.
    if predicted_classes.shape != target_class.shape:  # 예측과 정답 클래스 벡터 길이가 같은지 확인합니다.
        raise ValueError("예측과 정답 클래스 벡터 모양이 같아야 합니다.")  # 다른 클래스 체계를 섞는 실수를 예방합니다.

    responsible_index, ious = select_responsible_predictor(predicted_boxes, target_box)  # 최고 IoU 예측기를 책임자로 정합니다.
    responsible_box = predicted_boxes[responsible_index]  # 책임 예측기의 네 좌표만 선택합니다.

    xy_loss = LAMBDA_COORD * np.sum((responsible_box[:2] - target_box[:2]) ** 2)  # 중심 좌표 차이를 제곱·합산하고 λ_coord를 곱합니다.
    predicted_sqrt_wh = np.sqrt(responsible_box[2:])  # 책임 예측 폭과 높이에 제곱근을 적용합니다.
    target_sqrt_wh = np.sqrt(target_box[2:])  # 정답 폭과 높이에도 같은 변환을 적용합니다.
    wh_loss = LAMBDA_COORD * np.sum((predicted_sqrt_wh - target_sqrt_wh) ** 2)  # 변환 공간의 크기 오차를 제곱·합산합니다.

    object_confidence_target = ious[responsible_index]  # 논문 정의에 따라 객체 신뢰도 목표를 현재 IoU로 둡니다.
    object_confidence_loss = (predicted_confidences[responsible_index] - object_confidence_target) ** 2  # 책임 신뢰도와 IoU 목표의 제곱 오차입니다.
    non_responsible_mask = np.arange(predicted_boxes.shape[0]) != responsible_index  # 책임자가 아닌 박스 위치를 True로 표시합니다.
    no_object_confidence_loss = LAMBDA_NOOBJ * np.sum(predicted_confidences[non_responsible_mask] ** 2)  # 비책임 박스 신뢰도를 0으로 유도합니다.
    classification_loss = np.sum((predicted_classes - target_class) ** 2)  # 객체 셀의 클래스별 제곱 오차를 합산합니다.

    components = {  # 사람이 각 항의 크기를 진단할 수 있도록 딕셔너리로 이름과 값을 묶습니다.
        "xy": float(xy_loss),  # NumPy 스칼라를 일반 float로 변환해 저장합니다.
        "wh": float(wh_loss),  # 폭·높이 손실을 float로 저장합니다.
        "object_confidence": float(object_confidence_loss),  # 객체 신뢰도 손실을 저장합니다.
        "no_object_confidence": float(no_object_confidence_loss),  # 비객체 신뢰도 손실을 저장합니다.
        "classification": float(classification_loss),  # 클래스 손실을 저장합니다.
    }
    components["total"] = float(sum(components.values()))  # 앞의 다섯 항을 더해 전체 손실을 별도 키에 기록합니다.
    components["responsible_index"] = responsible_index  # 디버깅을 위해 책임 예측기 인덱스도 결과에 포함합니다.
    components["responsible_iou"] = float(object_confidence_target)  # 신뢰도 목표가 된 IoU도 결과에 포함합니다.
    return components  # 계산된 구성 요소 딕셔너리를 호출자에게 반환합니다.


predicted_confidences = np.array([0.78, 0.35], dtype=np.float64)  # 두 예측기의 현재 신뢰도를 정의합니다.
predicted_classes = np.array([0.10, 0.75, 0.15], dtype=np.float64)  # 세 클래스의 조건부 예측을 정의합니다.
target_class = np.array([0.0, 1.0, 0.0], dtype=np.float64)  # 두 번째 클래스가 정답인 원-핫 벡터를 정의합니다.
losses = yolo_v1_object_cell_loss(  # 여러 인수를 이름으로 전달해 호출 의미를 분명히 합니다.
    predicted_boxes=predicted_boxes,  # 앞에서 정의한 두 박스 예측을 전달합니다.
    predicted_confidences=predicted_confidences,  # 박스별 신뢰도를 전달합니다.
    predicted_classes=predicted_classes,  # 셀의 클래스 예측을 전달합니다.
    target_box=target_box,  # 정답 박스를 전달합니다.
    target_class=target_class,  # 정답 클래스 벡터를 전달합니다.
)

for name, value in losses.items():  # 딕셔너리의 키와 값을 순서대로 순회합니다.
    print(f"{name:>24}: {value}")  # 이름을 24칸 오른쪽 정렬해 구성 요소를 읽기 쉽게 출력합니다.

assert losses["responsible_index"] == 0  # 손실 함수 안에서도 첫 예측기가 책임자로 선택됐는지 확인합니다.
assert losses["total"] >= 0.0  # 제곱 오차의 가중합은 음수가 될 수 없습니다.

### 구현 주의점

원 논문은 제곱근 폭·높이 항을 사용합니다. 실제 네트워크의 제약되지 않은 출력이 음수가 될 가능성까지 포함하는 재현 구현은 논문 수식과 원 코드의 부호 처리 방식을 별도로 대조해야 합니다. 이 교육용 함수는 입력 검증으로 양의 크기만 받습니다. 또한 객체가 전혀 없는 셀에서는 모든 B개 신뢰도가 `λ_noobj` 항에 들어가며, 클래스·좌표 손실은 계산하지 않습니다.

## 3. Precision-Recall과 VOC 2007 11-point AP

검출 후보를 점수 내림차순으로 정렬한 뒤 TP/FP를 누적합니다. recall 0.0, 0.1, …, 1.0의 각 지점에서 그 이상의 recall이 갖는 최대 precision을 구해 평균하면 11-point AP입니다. 실제 검출 평가에서는 먼저 클래스별로 IoU 기준에 따라 TP/FP를 매칭해야 하며, 여기서는 그 매칭 결과가 `is_true_positive`로 주어졌다고 가정합니다.

In [ ]:
def precision_recall_curve(scores, is_true_positive, total_ground_truth):  # 점수와 TP 여부에서 누적 precision과 recall을 계산합니다.
    scores = np.asarray(scores, dtype=np.float64)  # 점수를 실수 배열로 통일합니다.
    is_true_positive = np.asarray(is_true_positive, dtype=bool)  # TP 표식을 True/False 배열로 통일합니다.
    if scores.shape != is_true_positive.shape:  # 각 후보마다 점수와 TP 표식이 하나씩 있는지 확인합니다.
        raise ValueError("scores와 is_true_positive의 모양이 같아야 합니다.")  # 길이가 다른 잘못된 평가 입력을 거부합니다.
    if total_ground_truth <= 0:  # recall의 분모인 정답 수가 양수인지 확인합니다.
        raise ValueError("total_ground_truth는 양수여야 합니다.")  # 0으로 나누는 무의미한 평가를 막습니다.

    order = np.argsort(scores)[::-1]  # 점수 오름차순 인덱스를 만든 뒤 뒤집어 내림차순으로 정렬합니다.
    sorted_tp = is_true_positive[order].astype(np.int64)  # 정렬된 불리언을 누적 가능한 0/1 정수로 바꿉니다.
    sorted_fp = 1 - sorted_tp  # 모든 후보가 TP 또는 FP라는 가정에서 1-TP로 FP를 만듭니다.
    cumulative_tp = np.cumsum(sorted_tp)  # cumsum으로 순위별 누적 TP 수를 계산합니다.
    cumulative_fp = np.cumsum(sorted_fp)  # 순위별 누적 FP 수도 계산합니다.
    precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1)  # 지금까지의 검출 중 맞은 비율을 계산합니다.
    recall = cumulative_tp / float(total_ground_truth)  # 전체 정답 중 지금까지 찾은 비율을 계산합니다.
    return precision, recall, order  # 곡선 값과 실제 정렬 순서를 함께 반환합니다.


def voc2007_11_point_ap(precision, recall):  # VOC 2007의 11개 recall 지점 보간 AP를 계산합니다.
    recall_levels = np.linspace(0.0, 1.0, 11)  # linspace로 0부터 1까지 같은 간격의 11개 값을 만듭니다.
    interpolated_precisions = []  # 각 recall 지점의 보간 precision을 저장할 빈 리스트입니다.

    for level in recall_levels:  # 0.0, 0.1, …, 1.0을 차례대로 평가합니다.
        eligible = precision[recall >= level]  # 현재 지점 이상의 recall을 달성한 precision만 선택합니다.
        best_precision = float(np.max(eligible)) if eligible.size else 0.0  # 후보가 있으면 최댓값, 없으면 0을 사용합니다.
        interpolated_precisions.append(best_precision)  # 이 지점의 보간 precision을 목록에 추가합니다.

    return float(np.mean(interpolated_precisions)), recall_levels, np.array(interpolated_precisions)  # 11개 평균과 진단 배열을 반환합니다.


scores = np.array([0.95, 0.90, 0.80, 0.70, 0.60, 0.40], dtype=np.float64)  # 여섯 검출 후보의 점수를 정의합니다.
is_true_positive = np.array([True, False, True, True, False, True])  # IoU 매칭까지 끝난 후보별 TP/FP 결과를 정의합니다.
precision, recall, ranking = precision_recall_curve(scores, is_true_positive, total_ground_truth=5)  # 정답 객체가 총 다섯 개라고 두고 곡선을 계산합니다.
average_precision, recall_levels, interpolated = voc2007_11_point_ap(precision, recall)  # 11-point AP와 보간값을 계산합니다.

print("정렬 인덱스:", ranking)  # 점수 내림차순의 원래 후보 인덱스를 확인합니다.
print("precision:", precision)  # 후보를 하나씩 추가할 때의 정밀도를 출력합니다.
print("recall:", recall)  # 후보를 하나씩 추가할 때의 재현율을 출력합니다.
print("보간 precision:", interpolated)  # 11개 recall 지점의 최대 precision을 출력합니다.
print(f"VOC 2007 11-point AP: {average_precision:.4f}")  # 최종 AP를 소수점 넷째 자리까지 출력합니다.

assert np.all(np.diff(recall) >= 0.0)  # 후보가 늘어날 때 누적 recall은 감소하지 않아야 합니다.
assert 0.0 <= average_precision <= 1.0  # AP는 확률적 품질 지표이므로 0과 1 사이여야 합니다.

## 실무 배포 전 릴리스 게이트

다음 항목을 자동 테스트와 실제 장치 측정으로 통과시킨 뒤 배포하세요.

1. **좌표 계약:** 학습·변환·C++ 전처리·후처리의 RGB/BGR, NCHW/NHWC, 정규화, letterbox와 좌표 복원 규칙이 동일한가?
2. **수치 동등성:** 같은 입력에서 Python 기준 출력과 ONNX/TensorRT/OpenVINO 출력의 허용 오차가 정해져 있는가?
3. **정확도 회귀:** 전체 mAP뿐 아니라 클래스별 AP, 작은 물체, 가림, 야간, 빈 장면 오탐을 평가했는가?
4. **실시간성:** 평균 FPS만이 아니라 p50/p95/p99 지연시간, 워밍업, 전처리·후처리와 메모리 복사를 포함했는가?
5. **안전 동작:** 낮은 신뢰도, 센서 끊김, 시간 초과, 모델 로드 실패 시 로봇/차량의 안전 상태가 정의돼 있는가?
6. **관측 가능성:** 모델 버전, 입력 품질, 후보 수, 억제 수, 지연시간과 오류 원인이 로그/메트릭으로 남는가?

### 다음 확장 과제

- 객체가 없는 셀을 포함한 `S×S×B` 배치 손실로 확장하세요.
- IoU 매칭부터 클래스별 AP와 mAP까지 하나의 평가기로 연결하세요.
- 같은 사전·후처리를 Python과 C++로 각각 구현해 골든 입력/출력 파일로 비교하세요.
- YOLO v1의 한계를 anchor, multi-scale prediction, IoU 계열 손실, decoupled head 같은 후속 설계와 비교하세요.